# 04 Data Preprocessing

This notebook contains **Step 4** of the workflow:

- change-feature creation
- missing-value imputation
- categorical encoding
- train/test split preparation

This notebook expects the `model_features` table created in **03_Feature_Engineering.ipynb**.


In [7]:
# Ensure this notebook can run independently by connecting to DuckDB
import duckdb
import os
import pandas as pd # often used alongside

from utils import get_db_connection
con = get_db_connection()


Connected to DuckDB at: /Users/sameer/Documents/DataScience_Capstone_Project/Capstone_Healthcare_Decision_Intelligence_Agent/dataset/hf_project.duckdb


# Step 4: Data Preprocessing — Feature Engineering, Imputation, Encoding, and Train/Test Split


In [8]:
import pandas as pd
import numpy as np

df = con.execute("SELECT * FROM model_features").fetchdf()

# 1. Add lab CHANGE features (last - first)
LABS = [
    'creatinine', 'urea_nitrogen', 'sodium', 'potassium', 'glucose',
    'hemoglobin', 'white_blood_cells', 'platelet_count', 'bicarbonate',
    'calcium_total', 'inrpt', 'ptt', 'troponin_t',
    'creatine_kinase_mb_isoenzyme'
]

for lab in LABS:
    df[f'{lab}_change'] = df[f'{lab}_last'] - df[f'{lab}_first']

print(f"After adding change features: {df.shape[0]} rows, {df.shape[1]} columns")

#  2. Fill missing values
# Numerical columns: fill with median
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
num_cols.remove('readmitted_30d')  # don't touch the label
num_cols.remove('hadm_id')
num_cols.remove('subject_id')

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

# Categorical columns: fill with 'UNKNOWN'
cat_cols = ['gender', 'admission_type', 'insurance', 'marital_status', 'race', 'discharge_location']
for col in cat_cols:
    df[col] = df[col].fillna('UNKNOWN')

print(f"\nMissing values after filling: {df.isnull().sum().sum()}")

# 3. Encode categorical columns
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True)

print(f"After encoding: {df_encoded.shape[0]} rows, {df_encoded.shape[1]} columns")

# 4. Train/Test Split
from sklearn.model_selection import train_test_split

# Separate features and label
drop_cols = ['hadm_id', 'subject_id', 'readmitted_30d']
X = df_encoded.drop(columns=drop_cols)
y = df_encoded['readmitted_30d']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n Final Split ")
print(f"Training set: {X_train.shape[0]} rows, {X_train.shape[1]} features")
print(f"Test set:     {X_test.shape[0]} rows, {X_test.shape[1]} features")
print(f"\nTrain label distribution:")
print(y_train.value_counts().to_string())
print(f"\nTest label distribution:")
print(y_test.value_counts().to_string())

# Save splits to DuckDB
con.execute("CREATE OR REPLACE TABLE X_train AS SELECT * FROM X_train")
con.execute("CREATE OR REPLACE TABLE X_test AS SELECT * FROM X_test")
y_train_df = y_train.to_frame()
con.execute("CREATE OR REPLACE TABLE y_train AS SELECT * FROM y_train_df")
y_test_df = y_test.to_frame()
con.execute("CREATE OR REPLACE TABLE y_test AS SELECT * FROM y_test_df")
print("Saved X_train, X_test, y_train, y_test to DuckDB")


After adding change features: 4508 rows, 85 columns

Missing values after filling: 0
After encoding: 4508 rows, 140 columns

 Final Split 
Training set: 3606 rows, 137 features
Test set:     902 rows, 137 features

Train label distribution:
readmitted_30d
0    2829
1     777

Test label distribution:
readmitted_30d
0    708
1    194
Saved X_train, X_test, y_train, y_test to DuckDB


Prepares the dataset for modeling through four preprocessing steps:
1. **Lab change features** — computes `last - first` values for all 14 labs to capture patient trajectory during hospitalization.
2. **Missing value imputation** — fills numerical columns with median values and categorical columns with 'UNKNOWN'.
3. **One-hot encoding** — converts 6 categorical features (gender, admission type, insurance, marital status, race, discharge location) into binary columns using `pd.get_dummies`.
4. **Stratified train/test split** — 80/20 split preserving the class ratio (~21.5% readmitted in both sets).

Final dimensions: Training set (3,606 rows × 137 features), Test set (902 rows × 137 features).

In [9]:
con.close()